# Большие языковые модели: семантический поиск по литературе

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Большие языковые модели».

## Что делает метод

Большие языковые модели работают с текстом на естественном языке. Одно из самых востребованных научных применений — **семантический поиск по литературе**: поиск релевантных работ по смыслу запроса, а не по точному совпадению ключевых слов. Это резко ускоряет вход в незнакомую область и поиск работ, сформулированных другими терминами.

В основе семантического поиска лежат **векторные представления (эмбеддинги)**: модель переводит каждый текст в числовой вектор так, что близкие по смыслу тексты получают близкие векторы. Этот же приём — фундамент технологии RAG (генерация с опорой на поиск), которая снижает долю выдуманных фактов.

В этом блокноте мы построим семантический поиск по небольшой коллекции аннотаций и сравним его с поиском по ключевым словам. Блокнот не требует API-ключей и платных сервисов. Расчётное время прохождения — около пяти минут.

> Полноценные диалоговые модели вызываются через API провайдера (в том числе отечественные сервисы) и требуют ключа доступа; ключи не размещаются в общедоступном блокноте. Здесь показана воспроизводимая, открытая часть рабочего процесса — построение и использование эмбеддингов.

## Интуиция за методом

Большие языковые модели — это, по сути, очень начитанные системы, обученные на огромных объёмах текста. Они видели столько научных статей, учебников и кода, что у них сформировалось «понимание» того, какие слова и идеи стоят рядом.

Их ключевой компонент — **эмбеддинги**: способность переводить любой текст в числовой вектор так, чтобы смысловая близость текстов превращалась в геометрическую близость векторов. Фраза «опухоль молочной железы» и фраза «карцинома груди» окажутся рядом в этом пространстве, даже если слова разные — потому что контексты, в которых они встречаются, похожи.

Именно на эмбеддингах строится **семантический поиск** — то, что мы будем делать в этом блокноте. Вместо поиска по точным словам мы ищем по смыслу: запрос «как нейросети помогают изучать снимки тканей» найдёт аннотацию про «сегментацию клеток на микрофотографиях», потому что их векторы окажутся близко.

**Ключевые термины, которые встретятся в блокноте:**

- **Эмбеддинг (embedding, векторное представление)** — числовой вектор, кодирующий смысловое содержание текста; похожие тексты имеют близкие векторы.
- **Косинусная близость** — мера сходства двух векторов: 1 означает полное совпадение направлений (близкий смысл), 0 — перпендикулярность (нет связи), -1 — противоположность.
- **RAG (retrieval-augmented generation, генерация с опорой на поиск)** — подход, при котором языковая модель отвечает не «по памяти», а опираясь на заранее найденные релевантные фрагменты из вашей коллекции. Это снижает риск выдуманных фактов.
- **PCA (анализ главных компонент)** — метод снижения размерности: сжимает многомерный вектор в две-три координаты для наглядной визуализации, сохраняя максимум различий между объектами.

## 1. Установка библиотек

Используем `sentence-transformers` для построения смысловых векторов. Если установка или загрузка модели не удалась (например, нет доступа к сети), блокнот автоматически переходит на запасной вариант — поиск по ключевым словам на основе TF-IDF из `scikit-learn`.

In [ ]:
%pip install -q sentence-transformers==3.0.1 scikit-learn==1.5.1 numpy==1.26.4 pandas==2.2.2 matplotlib==3.9.2

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Соберём небольшую коллекцию из двенадцати кратких аннотаций по разным научным темам. В реальной работе на этом месте была бы ваша личная библиотека статей. Аннотации намеренно сформулированы разными словами, чтобы показать преимущество смыслового поиска перед поиском по совпадению ключевых слов.

In [ ]:
import pandas as pd

abstracts = [
    "Анализ спутниковых снимков для оценки динамики таяния ледников в Арктике.",
    "Прогнозирование урожайности зерновых культур по метеорологическим данным.",
    "Машинное обучение для предсказания свойств новых кристаллических материалов.",
    "Сегментация клеток на снимках микроскопии для подсчёта и анализа морфологии.",
    "Статистический анализ клинических испытаний нового лекарственного препарата.",
    "Моделирование распространения инфекционного заболевания в популяции.",
    "Определение трёхмерной структуры белка по аминокислотной последовательности.",
    "Поиск экзопланет по колебаниям яркости звёзд в данных космического телескопа.",
    "Оценка экономических последствий изменения климата для сельского хозяйства.",
    "Распознавание видов растений по фотографиям листьев методами компьютерного зрения.",
    "Численное моделирование турбулентных течений жидкости в инженерных задачах.",
    "Обработка естественного языка для извлечения данных из научных публикаций.",
]
corpus = pd.DataFrame({"id": range(1, len(abstracts) + 1),
                       "аннотация": abstracts})
print(f"Аннотаций в коллекции: {len(corpus)}")
corpus

## 4. Применение метода

### Шаг 1. Построение эмбеддингов

Ячейка ниже переводит каждую аннотацию в числовой вектор. Если модель `sentence-transformers` доступна (требует интернет-соединения), используется нейросетевой энкодер — он понимает синонимы и парафразы. Если нет — используется классический TF-IDF: он хорошо работает при совпадении слов, но хуже на перефразированных запросах. Строка вывода покажет, какой режим активен.

> **Внимание:** при первом запуске скачивается модель эмбеддингов (~120 МБ).

In [ ]:
import numpy as np

method = None
try:
    from sentence_transformers import SentenceTransformer
    encoder = SentenceTransformer(
        "paraphrase-multilingual-MiniLM-L12-v2")
    doc_vectors = encoder.encode(corpus["аннотация"].tolist(),
                                 normalize_embeddings=True)
    method = "смысловые эмбеддинги"

    def embed(texts):
        return encoder.encode(list(texts), normalize_embeddings=True)
except Exception as err:
    print("Модель эмбеддингов недоступна, используем TF-IDF:", err)
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.preprocessing import normalize
    vectorizer = TfidfVectorizer()
    doc_matrix = vectorizer.fit_transform(corpus["аннотация"].tolist())
    doc_vectors = normalize(doc_matrix).toarray()
    method = "TF-IDF (поиск по ключевым словам)"

    def embed(texts):
        return normalize(vectorizer.transform(list(texts))).toarray()

print(f"Метод представления текста: {method}")
print(f"Размерность вектора: {doc_vectors.shape[1]}")

### Шаг 2. Семантический поиск по запросу

Функция `semantic_search` переводит запрос в вектор той же моделью и вычисляет **косинусную близость** между вектором запроса и векторами всех аннотаций. Документы ранжируются по убыванию близости.

Обратите внимание на запрос: «как нейросети помогают изучать снимки живых тканей». Ни одна из 12 аннотаций не содержит именно этих слов — но нейросетевой энкодер найдёт аннотацию о сегментации клеток на микрофотографиях, потому что смысл совпадает.

In [ ]:
def semantic_search(query, top_k=3):
    """Возвращает ближайшие по смыслу аннотации для текстового запроса."""
    query_vector = embed([query])[0]
    similarity = doc_vectors @ query_vector          # косинусная близость
    order = np.argsort(similarity)[::-1][:top_k]
    result = corpus.iloc[order].copy()
    result["близость"] = similarity[order].round(3)
    return result.reset_index(drop=True)


# Запрос сформулирован словами, которых нет ни в одной аннотации напрямую.
query = "как нейросети помогают изучать снимки живых тканей"
print(f"Запрос: {query}\n")
semantic_search(query)

### Шаг 3. Визуализация: релевантность и карта коллекции

Два графика ниже дают разные взгляды на результат поиска:
- **Левый**: сколько каждая аннотация «подходит» под запрос.
- **Правый**: карта всей коллекции в смысловом пространстве — какие темы соседствуют друг с другом.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 2, figsize=(14, 5.6))

# Близость аннотаций к запросу
query_vector = embed([query])[0]
similarity = doc_vectors @ query_vector
order = np.argsort(similarity)
labels = [f"#{corpus['id'].iloc[i]}" for i in order]
bar_colors = [VIZ["series"][1] if similarity[i] >= np.sort(similarity)[-3]
              else VIZ["series"][0] for i in order]
axes[0].barh(labels, similarity[order], color=bar_colors)
axes[0].set_title("Релевантность аннотаций запросу")
axes[0].set_xlabel("Смысловая близость к запросу")
axes[0].set_ylabel("Номер аннотации")
axes[0].grid(True, axis="x")

# Карта коллекции
coords = PCA(n_components=2, random_state=42).fit_transform(doc_vectors)
axes[1].scatter(coords[:, 0], coords[:, 1], s=70, color=VIZ["series"][0])
for i, row in corpus.iterrows():
    axes[1].annotate(f"#{row['id']}", (coords[i, 0], coords[i, 1]),
                     textcoords="offset points", xytext=(7, 4), fontsize=10)
axes[1].set_title("Карта коллекции аннотаций")
axes[1].set_xlabel("Ось 1")
axes[1].set_ylabel("Ось 2")

fig.tight_layout()
plt.show()

### Как читать эти графики

**График релевантности (левый):**
- Горизонтальные полосы — аннотации, отсортированные по близости к запросу.
- Длина полосы = косинусная близость (от 0 до 1). Выделенные (более насыщенные) полосы — топ-3 результата.
- Резкий перепад длин разделяет релевантные документы от нерелевантных. Если перепада нет — все документы примерно одинаково далеки от запроса (вероятно, запрос сформулирован слишком широко).

**Карта коллекции (правый):**
- Каждая точка — одна аннотация, спроецированная из многомерного пространства эмбеддингов на плоскость методом PCA.
- Точки, расположенные **близко**, близки по смыслу; расположенные **далеко** — на разные темы.
- Группы точек образуют тематические кластеры вашей коллекции. Изолированная точка — работа на уникальную тему, не имеющую аналогов в вашем наборе.
- Оси не имеют содержательной интерпретации — смысл несут только расстояния между точками.

## 5. Интерпретация результата

- **Смысловая близость** — число от 0 до 1: чем оно выше, тем ближе аннотация к запросу по содержанию. Семантический поиск находит релевантные работы, даже если они сформулированы другими словами; поиск по точному совпадению ключевых слов такие работы пропускает.
- **График релевантности**: выделенные аннотации — наиболее подходящие ответы на запрос. Резкий перепад длины столбцов отделяет релевантные работы от остальных.
- **Карта коллекции**: тексты одной тематики собираются рядом. Это помогает увидеть структуру личной библиотеки и обнаружить тематические пробелы.
- **Метод представления**: при работе через модель эмбеддингов поиск опирается на смысл; в режиме TF-IDF — на совпадение слов, поэтому качество на перефразированных запросах ниже. Строка вывода в разделе 4 показывает, какой режим использован.

Важно об ответственном применении: языковые модели формулируют связно, но могут уверенно выдумывать факты, ссылки и DOI. Семантический поиск возвращает реальные документы из вашей коллекции, и каждый результат можно проверить по первоисточнику — именно поэтому подход с опорой на поиск (RAG) надёжнее свободной генерации. Для воспроизводимости фиксируйте использованную модель и её версию.

## 6. Попробуйте на своих данных

Замените демонстрационную коллекцию своими аннотациями или фрагментами статей. Подготовьте таблицу, где каждая строка — отдельный текст (аннотация, абзац, заметка).

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже и укажите имя файла и название текстового столбца.
3. Выполните ячейки раздела 4, затем задавайте свои запросы функцией `semantic_search`.
4. Для больших коллекций (тысячи документов) применяют специализированные векторные хранилища; принцип поиска остаётся тем же.

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv("my_abstracts.csv")          # ваш файл
# text_column = "аннотация"                     # имя текстового столбца
#
# corpus = pd.DataFrame({"id": range(1, len(df) + 1),
#                        "аннотация": df[text_column].astype(str)})
# print(f"Текстов в коллекции: {len(corpus)}")
#
# Выполните ячейки раздела 4, затем задайте свой запрос:
# semantic_search("ваш запрос на естественном языке")

## 7. Поэкспериментируйте сами

**Эксперимент 1. Перефразированный запрос.**
Смените запрос: вместо «как нейросети помогают изучать снимки живых тканей» попробуйте «компьютерное зрение в биологии» или «deep learning for histology». Меняется ли список топ-3? Попробуйте запрос на английском — нейросетевой многоязычный энкодер должен найти те же русскоязычные аннотации.

**Эксперимент 2. Точный vs. семантический поиск.**
Введите запрос, содержащий ровно те слова, что в одной из аннотаций, например: «сегментация клеток». Сравните результат с результатом функции `semantic_search`. Затем попробуйте синоним без точного совпадения — «разметка клеток» — и посмотрите, сохраняется ли тот же результат.

**Эксперимент 3. Добавьте свои аннотации.**
Вставьте в список `abstracts` 3–5 кратких описаний ваших собственных работ или работ из вашего обзора. Пересчитайте эмбеддинги и посмотрите, куда они попадут на карте коллекции.

**Эксперимент 4. Матрица близости.**
Вычислите матрицу попарной близости всей коллекции: `sim_matrix = doc_vectors @ doc_vectors.T` — и отобразите её как тепловую карту `plt.imshow(sim_matrix, cmap="Blues")`. Какие пары аннотаций наиболее похожи?

## Краткие выводы

- Семантический поиск через эмбеддинги находит релевантные документы **по смыслу**, а не по точным словам — это важно, когда терминология в коллекции неоднородна.
- Качество зависит от модели: общеязыковые модели хуже на узкоспециальных терминах; для биомедицины, химии и физики ищите доменно-специфичные энкодеры.
- Подход RAG (поиск + генерация) надёжнее свободной генерации: модель отвечает по конкретным найденным источникам, которые можно проверить.
- Карта коллекции — полезный инструмент для обнаружения тематических пробелов в вашей библиотеке.
- Обязательно проверяйте каждое конкретное утверждение языковой модели по первоисточнику: модели уверенно формулируют, но могут выдумывать факты и ссылки.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике релевантности аннотация #4 («Сегментация клеток на снимках микроскопии...») получила наивысшую косинусную близость к запросу «как нейросети помогают изучать снимки живых тканей», хотя слова «нейросети» и «живые ткани» в ней отсутствуют. Что именно обеспечило это совпадение, и как этот результат изменится при использовании TF-IDF вместо нейросетевых эмбеддингов?
2. Исследователь задаёт запрос функции `semantic_search`, получает результат с косинусной близостью 0.82 и использует эту аннотацию как подтверждение факта в статье. Назовите одно принципиальное ограничение семантического поиска, которое делает такое использование ненадёжным.
3. Вы хотите снизить долю выдуманных фактов при генерации текста языковой моделью по вашей коллекции статей. Объясните, почему подход RAG (генерация с опорой на поиск) решает эту проблему лучше, чем простое увеличение размера модели.

<details>
<summary>Показать ориентиры для ответов</summary>

1. Нейросетевая модель эмбеддингов обучена на миллиардах текстов и связала понятия «сегментация клеток», «микроскопия», «нейросети» и «анализ тканей» в близкие области пространства эмбеддингов. TF-IDF не найдёт эту аннотацию, поскольку ни одно слово запроса не совпадает со словами аннотации буквально.
2. Семантический поиск ранжирует по близости смысла, но не проверяет фактическую точность содержимого документа. Высокая косинусная близость означает лишь тематическое родство, а не то, что факт из аннотации верен, — для подтверждения факта необходимо обратиться к полному первоисточнику.
3. RAG заземляет генерацию на конкретных найденных фрагментах вашей коллекции: модель получает текст источника как явный контекст в промпте и отвечает по нему, а не по параметрам, выученным при предобучении. Увеличение размера модели улучшает связность и знания общего характера, но не устраняет галлюцинации в специализированной области — модель по-прежнему «не знает» ваших конкретных документов.
</details>
